# Account Service

Account creation is a system boundary — it involves handle validation, invite policy,
DID generation, and session token issuance.

**What you'll learn:**
- Handle validation rules and format checking
- Invite code lifecycle (generate, consume, track)
- Account storage and lookup by DID
- Session creation and token refresh

**Estimated Time:** 20-25 minutes

## Step 1: Handle Validation

Every account starts with a handle. ATProto services checks format rules
before the handle reaches the service layer.

In [ ]:
@interface HandleValidator : NSObject
- (BOOL)isValidHandle:(NSString *)handle;
@end

@implementation HandleValidator
- (BOOL)isValidHandle:(NSString *)handle {
    if (handle == nil || [handle length] < 3) return NO;
    if ([handle length] > 253) return NO;
    if (![handle containsString:@"."]) return NO;
    if ([handle containsString:@" "]) return NO;
    return YES;
}
@end

HandleValidator *hv = [[HandleValidator alloc] init];
NSLog(@"alice.bsky.social: %d", [hv isValidHandle:@"alice.bsky.social"]);
NSLog(@"nodot: %d", [hv isValidHandle:@"nodot"]);
NSLog(@"with space.com: %d", [hv isValidHandle:@"with space.com"]);

## Step 2: Invite Code Store

ATProto supports invite-only registration. `InviteCodeStore` manages code lifecycle:
generate codes, consume them on account creation, track which codes remain.

In [ ]:
@interface InviteCodeStore : NSObject
@property (nonatomic, strong) NSMutableArray *codes;
- (NSString *)generateCode;
- (BOOL)useCode:(NSString *)code;
- (int)remainingCount;
@end

@implementation InviteCodeStore
- (instancetype)init {
    self = [super init];
    if (self) { _codes = [NSMutableArray array]; }
    return self;
}
- (NSString *)generateCode {
    // Simplified: real codes are cryptographically random
    int n = [_codes count] + 1;
    NSString *code = [NSString stringWithFormat:@"invite-%d", n];
    [_codes addObject:code];
    NSLog(@"Generated: %@", code);
    return code;
}
- (BOOL)useCode:(NSString *)code {
    if (code == nil) return NO;
    for (int i = 0; i < [_codes count]; i++) {
        if ([[_codes objectAtIndex:i] isEqualToString:code]) {
            [_codes removeObjectAtIndex:i];
            return YES;
        }
    }
    return NO;
}
- (int)remainingCount {
    return [_codes count];
}
@end

InviteCodeStore *ics = [[InviteCodeStore alloc] init];
NSString *code1 = [ics generateCode];
NSString *code2 = [ics generateCode];
NSLog(@"Remaining: %d", [ics remainingCount]);
NSLog(@"Use code1: %d", [ics useCode:code1]);
NSLog(@"Use again: %d", [ics useCode:code1]);
NSLog(@"Remaining: %d", [ics remainingCount]);

## Step 3: Account Store

Accounts are stored by DID. account creation
validates the handle, checks the invite code, generates a DID, and persists the account.

In [ ]:
@interface AccountStore : NSObject
@property (nonatomic, strong) NSMutableDictionary *accounts;
@property (nonatomic, strong) HandleValidator *validator;
@property (nonatomic, strong) InviteCodeStore *inviteStore;
@property (nonatomic, assign) int nextDidNum;
- (NSDictionary *)createAccountForEmail:(NSString *)email
                               password:(NSString *)password
                                handle:(NSString *)handle
                            inviteCode:(NSString *)inviteCode;
- (NSDictionary *)getAccountForDid:(NSString *)did;
@end

@implementation AccountStore
- (instancetype)init {
    self = [super init];
    if (self) {
        _accounts = [NSMutableDictionary dictionary];
        _validator = [[HandleValidator alloc] init];
        _inviteStore = [[InviteCodeStore alloc] init];
        _nextDidNum = 1;
    }
    return self;
}
- (NSDictionary *)createAccountForEmail:(NSString *)email
                               password:(NSString *)password
                                handle:(NSString *)handle
                            inviteCode:(NSString *)inviteCode {
    // Validate handle
    if (![_validator isValidHandle:handle]) {
        NSLog(@"Invalid handle: %@", handle);
        return nil;
    }
    // Check invite code
    if (![_inviteStore useCode:inviteCode]) {
        NSLog(@"Invalid invite code: %@", inviteCode);
        return nil;
    }
    // Generate DID
    NSString *did = [NSString stringWithFormat:@"did:plc:acct%d", _nextDidNum];
    _nextDidNum = _nextDidNum + 1;
    // Store account
    NSDictionary *account = @{
        @"handle": handle,
        @"email": email,
        @"did": did,
        @"status": @"active"
    };
    [_accounts setObject:account forKey:did];
    NSLog(@"Account created: %@ -> %@", handle, did);
    return account;
}
- (NSDictionary *)getAccountForDid:(NSString *)did {
    return [_accounts objectForKey:did];
}
@end

## Step 4: Session Manager

After account creation, services issues a session with access and refresh tokens.
The `JWTMinter` signs tokens; here we use simplified token strings.

In [ ]:
@interface SessionManager : NSObject
@property (nonatomic, strong) NSMutableDictionary *sessions;
@property (nonatomic, assign) int tokenCounter;
- (NSDictionary *)createSessionForDid:(NSString *)did handle:(NSString *)handle;
- (NSDictionary *)refreshSession:(NSString *)refreshToken;
- (void)invalidateSessionsForDid:(NSString *)did;
@end

@implementation SessionManager
- (instancetype)init {
    self = [super init];
    if (self) {
        _sessions = [NSMutableDictionary dictionary];
        _tokenCounter = 1;
    }
    return self;
}
- (NSDictionary *)createSessionForDid:(NSString *)did handle:(NSString *)handle {
    NSString *accessJwt = [NSString stringWithFormat:@"access-%d", _tokenCounter];
    NSString *refreshJwt = [NSString stringWithFormat:@"refresh-%d", _tokenCounter];
    _tokenCounter = _tokenCounter + 1;
    NSDictionary *session = @{
        @"accessJwt": accessJwt,
        @"refreshJwt": refreshJwt,
        @"handle": handle,
        @"did": did
    };
    [_sessions setObject:session forKey:did];
    NSLog(@"Session created for %@", did);
    return session;
}
- (NSDictionary *)refreshSession:(NSString *)refreshToken {
    // Simplified: find session by refresh token, issue new access token
    for (NSString *did in _sessions) {
        NSDictionary *s = [_sessions objectForKey:did];
        if ([[s objectForKey:@"refreshJwt"] isEqualToString:refreshToken]) {
            NSString *newAccess = [NSString stringWithFormat:@"access-%d", _tokenCounter];
            _tokenCounter = _tokenCounter + 1;
            NSMutableDictionary *updated = [NSMutableDictionary dictionaryWithDictionary:s];
            [updated setObject:newAccess forKey:@"accessJwt"];
            [_sessions setObject:updated forKey:did];
            NSLog(@"Refreshed session for %@", did);
            return updated;
        }
    }
    NSLog(@"Refresh token not found");
    return nil;
}
- (void)invalidateSessionsForDid:(NSString *)did {
    [_sessions removeObjectForKey:did];
    NSLog(@"Sessions invalidated for %@", did);
}
@end

## Step 5: Full Account Lifecycle

Wire everything together: create an invite code, create an account, issue a session, refresh it.

In [ ]:
// Full lifecycle
AccountStore *store = [[AccountStore alloc] init];
SessionManager *sessions = [[SessionManager alloc] init];

// Generate an invite code
NSString *code = [store.inviteStore generateCode];

// Create account
NSDictionary *acct = [store createAccountForEmail:@"alice@example.com"
                                          password:@"secret"
                                           handle:@"alice.bsky.social"
                                       inviteCode:code];
NSString *did = [acct objectForKey:@"did"];

// Create session
NSDictionary *session = [sessions createSessionForDid:did handle:@"alice.bsky.social"];
NSLog(@"Access: %@", [session objectForKey:@"accessJwt"]);

// Refresh session
NSString *refresh = [session objectForKey:@"refreshJwt"];
NSDictionary *refreshed = [sessions refreshSession:refresh];
NSLog(@"New access: %@", [refreshed objectForKey:@"accessJwt"]);

// Verify account exists
NSDictionary *lookup = [store getAccountForDid:did];
NSLog(@"Account handle: %@", [lookup objectForKey:@"handle"]);

## Exercise

Add a `deleteAccount:` method to `AccountStore` that removes the account and invalidates
its sessions. The method should take a DID and return YES on success.

Hint: remove the account from the `_accounts` dictionary and call
`[sessionManager invalidateSessionsForDid:]`.

In [ ]:
// Exercise: implement deleteAccount:
// Your code here...

## Solution

In [ ]:
// Solution: deleteAccount removes from store and invalidates sessions
@interface AccountStore2 : NSObject
@property (nonatomic, strong) NSMutableDictionary *accounts;
@property (nonatomic, strong) SessionManager *sessionManager;
- (BOOL)deleteAccount:(NSString *)did;
@end

@implementation AccountStore2
- (BOOL)deleteAccount:(NSString *)did {
    if ([_accounts objectForKey:did] == nil) {
        NSLog(@"Account not found: %@", did);
        return NO;
    }
    [_accounts removeObjectForKey:did];
    [_sessionManager invalidateSessionsForDid:did];
    NSLog(@"Account deleted: %@", did);
    return YES;
}
@end

// Test it
AccountStore2 *store2 = [[AccountStore2 alloc] init];
store2.accounts = [NSMutableDictionary dictionary];
store2.sessionManager = [[SessionManager alloc] init];
[store2.accounts setObject:@{@"handle": @"bob.bsky.social"} forKey:@"did:plc:test"];
[store2.sessionManager createSessionForDid:@"did:plc:test" handle:@"bob.bsky.social"];
NSLog(@"Delete: %d", [store2 deleteAccount:@"did:plc:test"]);
NSLog(@"Delete again: %d", [store2 deleteAccount:@"did:plc:test"]);

## Next Steps

You have learned about ATProto's core data structures and protocols. Continue to explore how these concepts apply when building a PDS.